# Data collection

In [ ]:
pip install mediapipe opencv-python numpy onnx onnxruntime

# Data collection

In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import json
import os

# Constants
folder = "Dataset/landmarks/T"
file_path = f"{folder}/landmarks.json"
counter = 0
landmarks_data = []

os.makedirs(folder, exist_ok=True)

# Load existing data if available
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        try:
            landmarks_data = json.load(f)
        except json.JSONDecodeError:
            landmarks_data = []  # Handle empty or corrupted JSON

# Initialize
cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands_detector = mp_hands.Hands(min_detection_confidence=0.7, max_num_hands=1)

while True:
    success, img = cap.read()
    
    if not success:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands_detector.process(img_rgb)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        
        cv2.putText(img, "Hand Detected - Press 's' to save", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow("Image", img)
    key = cv2.waitKey(1)
    
    # Save landmarks when 's' is pressed
    if key == ord('s') and results.multi_hand_landmarks:
        landmarks = results.multi_hand_landmarks[0]
        features = []
        for lm in landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
        
        landmarks_data.append(features)
        counter += 1
        print(f"Saved landmark sample {counter}")
    
    # Exit on 'Esc'
    if key == 27:
        break

# Save updated data (append mode)
with open(file_path, 'w') as f:
    json.dump(landmarks_data, f, indent=4)

cap.release()
cv2.destroyAllWindows()
print(f"Total samples saved this session: {counter}")
print(f"Total samples stored in file now: {len(landmarks_data)}")


# Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import json
import os

# --- 1. Constants & Configuration ---

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths and Parameters
DATASET_PATH = "Dataset/landmarks"
MODEL_SAVE_PATH = "landmark_model.pth"
CLASSES_SAVE_PATH = "landmark_classes.json"
BATCH_SIZE = 32
EPOCHS = 150  # MLPs train very fast, so more epochs are fine
LEARNING_RATE = 0.001
TEST_SPLIT_SIZE = 0.2  # Use 20% of data for validation

# --- 2. Load and Prepare Data ---

def load_data(dataset_path):
    """Loads all landmark.json files from subdirectories."""
    all_features = []
    all_labels = []
    class_to_idx = {}
    idx_to_class = {}
    
    current_idx = 0
    
    # Iterate over class folders (e.g., "A", "B")
    for class_name in os.listdir(dataset_path):
        class_path = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_path):
            continue

        # Add class to mapping
        if class_name not in class_to_idx:
            class_to_idx[class_name] = current_idx
            idx_to_class[current_idx] = class_name
            current_idx += 1
            
        label = class_to_idx[class_name]
        
        # Load the json file for this class
        json_path = os.path.join(class_path, 'landmarks.json')
        if not os.path.exists(json_path):
            print(f"Warning: No landmarks.json found in {class_path}")
            continue
            
        with open(json_path, 'r') as f:
            landmark_data = json.load(f)
            
        # Add each sample (list of 63 floats) to our dataset
        for sample in landmark_data:
            all_features.append(sample)
            all_labels.append(label)
            
    return (
        np.array(all_features, dtype=np.float32),
        np.array(all_labels, dtype=np.int64),
        class_to_idx,
        idx_to_class
    )

# Load all data from disk
features, labels, class_to_idx, idx_to_class = load_data(DATASET_PATH)

print(f"Classes found: {class_to_idx}")
print(f"Total samples: {len(features)}, Total labels: {len(labels)}")

if len(features) == 0:
    print("Error: No data loaded. Check your DATASET_PATH.")
    exit()

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    features, labels, 
    test_size=TEST_SPLIT_SIZE, 
    random_state=42, 
    stratify=labels  # Ensures balanced classes in train/val splits
)

print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")

# --- 3. Custom PyTorch Dataset ---

class LandmarkDataset(Dataset):
    """Custom dataset for landmark features."""
    def __init__(self, features, labels):
        # Convert to tensors
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create Dataset and DataLoader instances
train_dataset = LandmarkDataset(X_train, y_train)
val_dataset = LandmarkDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 4. Define the MLP Model ---

# Get input size (should be 63: 21 landmarks * 3 coords)
input_size = features.shape[1] 
num_classes = len(class_to_idx)

class MLP(nn.Module):
    def __init__(self, input_features, num_classes):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),  # Add dropout for regularization
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
            # No softmax here, as nn.CrossEntropyLoss includes it
        )
        
    def forward(self, x):
        return self.layers(x)

# Initialize model, loss, and optimizer
model = MLP(input_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 5. Training Loop ---

for epoch in range(EPOCHS):
    model.train()  # Set model to training mode
    running_loss = 0.0
    
    for batch_features, batch_labels in train_loader:
        # Send data to device
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    # --- Validation ---
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # No gradients needed for validation
        for batch_features, batch_labels in val_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            val_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    train_loss_epoch = running_loss / len(train_loader)
    val_loss_epoch = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss_epoch:.4f} | "
          f"Val Loss: {val_loss_epoch:.4f} | "
          f"Val Acc: {val_accuracy:.2f}%")

# --- 6. Save the Model and Class Mappings ---

# Save the model's weights
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

# Save the class-to-index mapping (CRITICAL for inference)
with open(CLASSES_SAVE_PATH, 'w') as f:
    json.dump(class_to_idx, f)
print(f"Class mapping saved to {CLASSES_SAVE_PATH}")

# ONNX Conversion

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import onnx
import onnxruntime
import json
import os

# --- 1. Configuration ---

# This is fixed by MediaPipe (21 landmarks * 3 coordinates)
INPUT_FEATURES = 63 

# Paths to your trained model and class mapping
PTH_MODEL_PATH = "landmark_model.pth"
CLASSES_JSON_PATH = "landmark_classes.json"
ONNX_MODEL_PATH = "landmark_model.onnx"

# Check if required files exist
if not os.path.exists(PTH_MODEL_PATH):
    print(f"Error: Model file not found at {PTH_MODEL_PATH}")
    print("Please run the landmark training script first.")
    exit()
    
if not os.path.exists(CLASSES_JSON_PATH):
    print(f"Error: Class mapping file not found at {CLASSES_JSON_PATH}")
    print("Please run the landmark training script first.")
    exit()

# --- 2. Re-create Your Model Architecture ---

# We must define the *exact* same model class as in the training script
class MLP(nn.Module):
    def __init__(self, input_features, num_classes):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        return self.layers(x)

# --- 3. Load Model and Weights ---

print("Loading model architecture and weights...")

# Load the class mapping to find out how many classes we have
with open(CLASSES_JSON_PATH, 'r') as f:
    class_to_idx = json.load(f)
num_classes = len(class_to_idx)
print(f"Found {num_classes} classes: {list(class_to_idx.keys())}")

# We use 'cpu' here as we don't need a GPU for the conversion process
device = torch.device("cpu")

# Initialize the model architecture
model = MLP(INPUT_FEATURES, num_classes).to(device)

# Load the trained weights
model.load_state_dict(torch.load(PTH_MODEL_PATH, map_location=device))

# IMPORTANT: Set the model to evaluation mode
model.eval()

# --- 4. Create a Dummy Input ---

# Create a dummy input tensor with the correct shape:
# (batch_size, num_features)
# This is different from the image model!
dummy_input = torch.randn(1, INPUT_FEATURES, device=device)
print(f"Using dummy input shape: {dummy_input.shape}")

# --- 5. Export to ONNX ---

print(f"Exporting model to {ONNX_MODEL_PATH}...")
torch.onnx.export(
    model,                  # The model to export
    dummy_input,            # The model's dummy input
    ONNX_MODEL_PATH,        # Where to save the model
    export_params=True,     # Store the trained weights
    opset_version=11,       # The ONNX version to use
    do_constant_folding=True, # Execute constant folding for optimization
    input_names=['input'],  # Name for the model's input
    output_names=['output'], # Name for the model's output
    dynamic_axes={          # Allows for variable batch sizes
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print("ONNX export complete.")

# --- 6. (Recommended) Verify the ONNX Model ---

print("Verifying the ONNX model...")

try:
    # Check that the model is well-formed
    onnx_model = onnx.load(ONNX_MODEL_PATH)
    onnx.checker.check_model(onnx_model)

    # Create an ONNX runtime session
    ort_session = onnxruntime.InferenceSession(ONNX_MODEL_PATH)

    # Helper function to convert torch tensor to numpy
    def to_numpy(tensor):
        return tensor.detach().cpu().numpy()

    # Get PyTorch model output
    with torch.no_grad():
        torch_out = model(dummy_input)

    # Get ONNX model output
    ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(dummy_input)}
    ort_outs = ort_session.run(None, ort_inputs)

    # Compare the outputs
    np.testing.assert_allclose(
        to_numpy(torch_out),
        ort_outs[0],
        rtol=1e-03,
        atol=1e-05
    )
    print("Verification successful! The ONNX model's output matches the PyTorch model's output.")

except Exception as e:
    print(f"Error during verification: {e}")


# Inference

In [ ]:
import cv2
import numpy as np
import onnxruntime
import mediapipe as mp
import json
import time

# --- 1. Helper Function: Softmax ---
def softmax(x):
    """Compute softmax values for a set of scores."""
    e_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return e_x / e_x.sum(axis=0)

# --- 2. Load ONNX Model and Class Labels ---
MODEL_PATH = "landmark_model.onnx"
CLASSES_JSON_PATH = "landmark_classes.json"

# Load the ONNX model
try:
    ort_session = onnxruntime.InferenceSession(MODEL_PATH)
    print("ONNX model loaded successfully.")
except Exception as e:
    print(f"Error loading ONNX model: {e}")
    print("Please make sure 'landmark_model.onnx' is in the same directory.")
    exit()

# Load the class labels
try:
    with open(CLASSES_JSON_PATH, 'r') as f:
        class_to_idx = json.load(f)
    # Create the inverse mapping (index to class name)
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    print(f"Loaded {len(idx_to_class)} classes: {list(idx_to_class.values())}")
except Exception as e:
    print(f"Error loading class labels: {e}")
    print(f"Please make sure '{CLASSES_JSON_PATH}' is in the same directory.")
    exit()

# --- 3. Initialize MediaPipe Hands ---
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
# Set static_image_mode=False for video stream
hands_detector = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)

# --- 4. Initialize Webcam ---
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

print("\nStarting inference... Press 'q' to quit.")

while True:
    success, img = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    # Flip the image horizontally for a "mirror" view
    img = cv2.flip(img, 1)

    # Convert BGR image to RGB for MediaPipe
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Process the image to find hand landmarks
    results = hands_detector.process(img_rgb)

    # --- 5. Process Hand Landmarks ---
    if results.multi_hand_landmarks:
        # Get the first (and only) detected hand
        hand_landmarks = results.multi_hand_landmarks[0]
        
        # Draw the landmarks on the image
        mp_drawing.draw_landmarks(
            img, hand_landmarks, mp_hands.HAND_CONNECTIONS
        )

        # --- 6. Extract Features (CRITICAL) ---
        # This MUST match the *exact* format from your data collection script
        features = []
        for lm in hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
        
        # Prepare data for ONNX model
        # 1. Convert to numpy array
        # 2. Ensure it's the correct float type (float32)
        # 3. Reshape to (1, 63) -> 1 batch, 63 features
        onnx_input = np.array(features, dtype=np.float32).reshape(1, -1)

        # --- 7. Run Inference ---
        # Get the input name from the ONNX model
        input_name = ort_session.get_inputs()[0].name
        
        # Start timer
        start_time = time.perf_counter()
        
        # Run the model
        ort_outs = ort_session.run(None, {input_name: onnx_input})
        
        # End timer
        end_time = time.perf_counter()
        
        inference_time_ms = (end_time - start_time) * 1000

        # --- 8. Post-process the Output ---
        # ort_outs[0] contains the raw output (logits) for our batch of 1
        logits = ort_outs[0][0]
        
        # Apply softmax to get probabilities
        probabilities = softmax(logits)
        
        # Get the predicted class index
        predicted_idx = np.argmax(probabilities)
        
        # Get the probability of that class
        confidence = probabilities[predicted_idx]
        
        # Get the class label
        predicted_label = idx_to_class[predicted_idx]

        # --- 9. Display Results on Screen ---
        # Create a black box for text
        cv2.rectangle(img, (10, 10), (350, 130), (0, 0, 0), -1)
        
        # Display Predicted Label
        text_label = f"PREDICTED: {predicted_label}"
        cv2.putText(img, text_label, (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display Probability
        text_prob = f"PROB: {confidence * 100:.2f}%"
        cv2.putText(img, text_prob, (20, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display Inference Time
        text_time = f"TIME: {inference_time_ms:.2f} ms"
        cv2.putText(img, text_time, (20, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    else:
        # If no hand is detected
        cv2.putText(img, "No hand detected", (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Show the final image
    cv2.imshow("Sign Language Inference", img)

    # Exit when 'esc' is pressed
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# --- 10. Cleanup ---
cap.release()
cv2.destroyAllWindows()
hands_detector.close()
print("Inference stopped.")